# Midterm Part A - thin Colab launcher
Sparse-checks-out only `midterm_a/` from your fork, installs it and runs `train.py`. All logic lives in the repo. Runtime -> Change runtime type -> **T4 GPU**. After a disconnect re-run the setup cell and the same command: runs resume from `last.ckpt`.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
import os

os.chdir("/content")

print("Current directory:", os.getcwd())

Current directory: /content


In [3]:
FORK_URL = "https://github.com/rafihi/representation-learning-course.git"
WORK = "/content/drive/MyDrive/ssl_midterm_a"      # checkpoints, results and the dataset cache live on Drive
!mkdir -p {WORK}/runs {WORK}/results {WORK}/data_cache
!rm -rf /content/repo && git clone --filter=blob:none --sparse {FORK_URL} /content/repo
%cd /content/repo
!git sparse-checkout set midterm_a
%cd midterm_a
!ln -s {WORK}/runs runs && ln -s {WORK}/results results && ln -s {WORK}/data_cache data_cache
!pip install -q -r requirements.txt

Cloning into '/content/repo'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 97 (delta 1), reused 0 (delta 0), pack-reused 92 (from 2)
Receiving objects: 100% (97/97), 14.07 KiB | 14.07 MiB/s, done.
Resolving deltas: 100% (38/38), done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 3 (from 1)
Receiving objects: 100% (4/4), 4.25 KiB | 4.25 MiB/s, done.
/content/repo
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 29 (delta 0), reused 0 (delta 0), pack-reused 27 (from 2)
Receiving objects: 100% (29/29), 27.32 KiB | 27.32 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/repo/midterm_a
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This 

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


## 0. Sanity checks (no downloads)

In [ ]:
!SSL_FAKE_DATA=1 pytest -q tests 2>&1 | tail -3

In [ ]:
!python train.py --config configs/a1_tune_dae.yaml --smoke --set out_dir=/tmp/smoke_a1 trainer.progress=false 2>&1 | tail -5

## 1. A2 - tuning (proxy) + reconstruction-vs-kNN figure

In [5]:
from pathlib import Path

out = Path("results/a1_tune_dae")

done = sorted(out.glob("trial_*/metrics.json"))

print(f"Successfully completed: {len(done)} / 20")

for p in sorted(out.glob("trial_*")):
    status = "DONE" if (p / "metrics.json").exists() else "INCOMPLETE"
    checkpoint = (p / "ckpt" / "last.ckpt").exists()
    print(p.name, status, "| Checkpoint:", checkpoint)

print("\nFinal plot exists:",
      (out / "recon_vs_knn.png").exists())

Successfully completed: 8 / 20
trial_00 DONE | Checkpoint: True
trial_01 DONE | Checkpoint: True
trial_02 DONE | Checkpoint: True
trial_03 DONE | Checkpoint: True
trial_04 DONE | Checkpoint: True
trial_05 DONE | Checkpoint: True
trial_06 DONE | Checkpoint: True
trial_07 DONE | Checkpoint: True
trial_08 INCOMPLETE | Checkpoint: False

Final plot exists: False


In [6]:
!python train.py --config configs/a1_tune_dae.yaml

Streaming output truncated to the last 5000 lines.
Epoch 0/14 ━━━━━━━━━━━━╺━━━━ 113/156 0:00:15 •        7.17it/s train_loss: 0.050
Epoch 0/14 ━━━━━━━━━━━━╺━━━━ 114/156 0:00:15 •        7.18it/s train_loss: 0.045
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 115/156 0:00:16 •        7.18it/s train_loss: 0.045
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 115/156 0:00:16 •        7.18it/s train_loss: 0.045
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 116/156 0:00:16 •        7.17it/s train_loss: 0.044
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 117/156 0:00:16 •        7.18it/s train_loss: 0.044
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 118/156 0:00:16 •        7.18it/s train_loss: 0.044
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 118/156 0:00:16 •        7.18it/s train_loss: 0.044
Epoch 0/14 ━━━━━━━━━━━━╸━━━━ 119/156 0:00:16 •        7.18it/s train_loss: 0.047
Epoch 0/14 ━━━━━━━━━━━━━╺━━━ 120/156 0:00:16 •        7.18it/s train_loss: 0.047
Epoch 0/14 ━━━━━━━━━━━━━╺━━━ 120/156 0:00:16 •        7.18it/s train_loss: 0.046
Epoch 0/14 ━━━━━━━━━━━━━╺━━━ 121/156 0:00:16 •        7.18

Open `results/a1_tune_dae/results.csv`, pick the trial with the best `knn_top1`, copy its `model.kind / level / lr / weight_decay` into `configs/a2_dae_final.yaml` (commit + push, re-run the setup cell), then:

In [7]:
import pandas as pd

df = pd.read_csv("results/a1_tune_dae/results.csv")

best = df.loc[df["knn_top1"].idxmax()]

print("BEST TRIAL PARAMETERS")
for col in [
    "trial",
    "model.kind",
    "model.level",
    "model.lr",
    "model.weight_decay",
    "knn_top1",
    "knn_top5"
]:
    print(f"{col}: {best[col]}")

BEST TRIAL PARAMETERS
trial: 13
model.kind: gaussian
model.level: 0.5016725270731012
model.lr: 0.0004165006165449
model.weight_decay: 4.255688904725068e-05
knn_top1: 6.6600002348423
knn_top5: 16.269999742507935


In [12]:
!grep "model:" configs/a2_dae_final.yaml

model: {kind: gaussian, level: 0.5016725270731012, lr: 0.0004165006165449, weight_decay: 0.00004255688904725068}


In [8]:
!SSL_FAKE_DATA=1 SSL_TINY_BACKBONE=1 python train.py --config configs/a2_dae_final.yaml --smoke --set out_dir=runs/ckpt_smoke

Seed set to 0
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
I0000 00:00:1795535859.113081   16366 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795535860.061383   16366 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795535863.183479   16366 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:2
1: `isinstance(treespec, LeafSpec)` is deprecated, use `i

In [9]:
from pathlib import Path

p = Path("/content/drive/MyDrive/ssl_midterm_a/runs/ckpt_smoke/ckpt/last.ckpt")

print("Checkpoint exists:", p.exists())

if p.exists():
    print("Size:", round(p.stat().st_size / 1024**2, 2), "MB")

Checkpoint exists: True
Size: 33.7 MB


In [ ]:
!python train.py --config configs/a2_dae_final.yaml

Streaming output truncated to the last 5000 lines.
Epoch 30/59 ━━━━━━━━━━━━━━╸━ 654/703 0:01:33 •        6.98it/s train_loss: 0.015
                                     0:00:08                   val_knn_top1:    
Epoch 30/59 ━━━━━━━━━━━━━━╸━ 655/703 0:01:33 •        6.98it/s train_loss: 0.015
                                     0:00:07                   val_knn_top1:    
Epoch 30/59 ━━━━━━━━━━━━━━╸━ 656/703 0:01:34 •        6.98it/s train_loss: 0.015
                                     0:00:07                   val_knn_top1:    
Epoch 30/59 ━━━━━━━━━━━━━━╸━ 657/703 0:01:34 •        6.98it/s train_loss: 0.015
                                     0:00:07                   val_knn_top1:    
Epoch 30/59 ━━━━━━━━━━━━━━╸━ 657/703 0:01:34 •        6.98it/s train_loss: 0.015
                                     0:00:07                   val_knn_top1:    
Epoch 30/59 ━━━━━━━━━━━━━━╸━ 658/703 0:01:34 •        6.98it/s train_loss: 0.013
                                     0:00:07              

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

root = Path('/content/drive/MyDrive/ssl_midterm_a')

print("=== A2 CHECKPOINT SEARCH ===")

for p in root.rglob("last.ckpt"):
    print("FOUND:", p)
    print("SIZE:", round(p.stat().st_size / 1024**2, 2), "MB")

Mounted at /content/drive
=== A2 CHECKPOINT SEARCH ===
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_19/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_18/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_17/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_16/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_15/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_14/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_13/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_12/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_11/ckpt/last.ckpt
SIZE: 159.91 MB
FOUND: /c

In [4]:
from pathlib import Path
import os

paths = [
    Path("/content/repo/midterm_a/runs"),
    Path("/content/repo/midterm_a/runs/dae_final"),
    Path("/content/drive/MyDrive/ssl_midterm_a/runs"),
    Path("/content/drive/MyDrive/ssl_midterm_a/runs/dae_final"),
    Path("/content/drive/MyDrive/ssl_midterm_a/runs/dae_final/ckpt")
]

for p in paths:
    print("\n", p)
    print("Exists:", p.exists())
    print("Is symlink:", p.is_symlink())

    if p.is_symlink():
        print("Target:", os.readlink(p))

    if p.exists() and p.is_dir():
        print("Contents:", [x.name for x in p.iterdir()])


 /content/repo/midterm_a/runs
Exists: True
Is symlink: True
Target: /content/drive/MyDrive/ssl_midterm_a/runs
Contents: ['dae_final']

 /content/repo/midterm_a/runs/dae_final
Exists: True
Is symlink: False
Contents: ['tb']

 /content/drive/MyDrive/ssl_midterm_a/runs
Exists: True
Is symlink: False
Contents: ['dae_final']

 /content/drive/MyDrive/ssl_midterm_a/runs/dae_final
Exists: True
Is symlink: False
Contents: ['tb']

 /content/drive/MyDrive/ssl_midterm_a/runs/dae_final/ckpt
Exists: False
Is symlink: False


In [5]:
from pathlib import Path
import os
import yaml

os.chdir("/content/repo/midterm_a")

root = Path("/content/drive/MyDrive/ssl_midterm_a")

# 1. חיפוש כל קובצי המודל ב-Drive
print("=== ALL CHECKPOINTS ===")

for p in root.rglob("*.ckpt"):
    print(p, round(p.stat().st_size / 1024**2, 2), "MB")

# 2. בדיקת הגדרות A2
with open("configs/a2_dae_final.yaml") as f:
    a2 = yaml.safe_load(f)

with open("configs/base_dae.yaml") as f:
    base = yaml.safe_load(f)

print("\n=== A2 CONFIG ===")
print("Output:", a2["out_dir"])
print("Max epochs:", a2["trainer"]["max_epochs"])
print("Resume:", base["resume"])
print("Checkpoint frequency:",
      base["trainer"]["ckpt_every"])

# 3. בדיקת תוכן תיקיית A2
out = root / "runs/dae_final"

print("\n=== A2 DRIVE CONTENTS ===")
for p in out.rglob("*"):
    print(p)

=== ALL CHECKPOINTS ===
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_19/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_18/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_17/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_16/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_15/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_14/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_13/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_12/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_11/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_10/ckpt/last.ckpt 159.91 MB
/content/drive/MyDrive/ssl_midterm_a/results/a1_tune_dae/trial_09/

In [6]:
%cd /content/repo/midterm_a
!ls -ld runs results data_cache
!ls -la runs/dae_final runs/dae_final/ckpt
!ls -la /content/drive/MyDrive/ssl_midterm_a/runs/dae_final

/content/repo/midterm_a
lrwxrwxrwx 1 root root 47 Nov 24 14:58 data_cache -> /content/drive/MyDrive/ssl_midterm_a/data_cache
lrwxrwxrwx 1 root root 44 Nov 24 14:58 results -> /content/drive/MyDrive/ssl_midterm_a/results
lrwxrwxrwx 1 root root 41 Nov 24 14:58 runs -> /content/drive/MyDrive/ssl_midterm_a/runs
ls: cannot access 'runs/dae_final/ckpt': No such file or directory
runs/dae_final:
total 12
drwx------ 3 root root 4096 Nov 24 13:36 .
drwx------ 3 root root 4096 Nov 23 20:59 ..
drwx------ 2 root root 4096 Nov 24 13:36 tb
total 12
drwx------ 3 root root 4096 Nov 24 13:36 .
drwx------ 3 root root 4096 Nov 23 20:59 ..
drwx------ 2 root root 4096 Nov 24 13:36 tb


In [ ]:
!python train.py --config configs/a3_vanilla_ae_final.yaml

## 2. A3 - frozen protocol vs. baselines (k-NN + linear probe)

In [ ]:
!python train.py --config configs/a4_eval.yaml

## 3. A3/A4 - fine-tuning and label efficiency

In [ ]:
!python train.py --config configs/a5_finetune.yaml

## 4. A4 - corruption curves

In [ ]:
!python train.py --config configs/a6_robustness.yaml

## 5. A5 - t-SNE, retrieval grid, per-class accuracy

In [ ]:
!python train.py --config configs/a7_interpret.yaml

## Final numbers on the TEST split (only once, after every decision is made)

In [ ]:
!python train.py --config configs/a4_eval.yaml --set split=test out_dir=results/final_a4_test
!python train.py --config configs/a5_finetune.yaml --set split=test out_dir=results/final_a5_test

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs